In [ ]:
import pandas as pd
import plotly.express as px
import geopandas as gpd
import plotly.graph_objects as go
import json

In [ ]:
#df = pd.read_csv('/home/jovyan/work/canc_air/01_data/data_octobre_2023/Pseudonymisation_provisoire_geocoded.csv', sep = ";")
df = pd.read_csv('H:/canc_air/data/test.csv', sep = ",")
df.drop('Unnamed: 0', axis=1, inplace=True)


gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x, df.y)).set_crs(epsg=4326)#.to_crs(epsg=2154) add ()


### Heatmap

In [ ]:
df['patient_count'] = 1

# Create a heatmap using plotly express
fig = px.density_mapbox(df, lat='y', lon='x', z='patient_count', 
                        radius=5,
                        center=dict(lat=48.8566, lon=2.3522), # Center on Paris
                        zoom=5
                        )

fig.update_layout(
    width=1400,  # Width of the figure
    height=800,  # Height of the figure
    mapbox=dict(
        #zoom=10,
        style="carto-positron",  # Map style (you can also use "open-street-map", "carto-positron", etc.)
    ),
    margin={"r":150, "t":30, "l":30, "b":30},  # Remove margins for a cleaner look
    coloraxis_colorbar=dict(
        title='Number of Patients',
        tickvals=[0, 15000, 30000, 45000, 60000],
        ticktext=['0', '15k', '30k', '45k', '60k']
    )
)


# Display the figure (this won't work in this environment, but the code is provided for reference)
fig.show()

fig.write_html("H:/canc_air/data/heatmap_figure.html")


In [ ]:
geo_df = gdf 
##Mise en forme du code iris pour obtenir code insee 
#geo_df["code_insee"] = geo_df.CODE_IRIS.astype(str).str[:5]

##Transformation du shp vers geojson pour cartographier les iris 
df_iris = gpd.read_file('/home/jovyan/work/canc_air/01_data/iris/iris.shp')
df_iris.to_file("/home/jovyan/work/canc_air/01_data/iris.geojson", driver='GeoJSON')

import json 
with open("/home/jovyan/work/canc_air/01_data/iris.geojson") as f:
    geojson_iris = json.load(f)
    
iris_geojson = gpd.read_file("/home/jovyan/work/canc_air/01_data/iris.geojson")


In [ ]:
print(f"Nombre de patients géocodés ne comportant pas de code Iris {geo_df.CODE_IRIS.isna().sum()}")
print(f"Proportion du nombre de patients ne comportant pas de code iris {geo_df.CODE_IRIS.isna().sum() / len(geo_df)}")

In [ ]:
## calcul du count groub by stocké dans une liste 
df_grouped = geo_df.groupby("CODE_IRIS").size().reset_index(name="count")

In [ ]:
iris_geojson

In [ ]:
# Merge the patient counts with the iris geojson data

iris_geojson['CODE_IRIS'] = iris_geojson['CODE_IRIS'].astype(int)
df_grouped['CODE_IRIS'] = df_grouped['CODE_IRIS'].astype(int)

choropleth_data = iris_geojson.merge(df_grouped, on='CODE_IRIS', how='left')

# Replace NaN with 0 in the Patient_Count column, as some IRIS codes might not have any patients
choropleth_data['count'] = choropleth_data['count'].fillna(0)

# Check the merged dataframe
choropleth_data[['CODE_IRIS', 'count']]

In [ ]:
choropleth_data['count'].isnull().any()

In [ ]:
choropleth_data

In [ ]:
import plotly.io as pio
pio.renderers.default = 'iframe'

In [ ]:
# Create a choropleth map using the merged data
fig = px.choropleth_mapbox(choropleth_data,
                           geojson= iris_geojson,
                           locations="CODE_IRIS",
                           locationmode='CODE_IRIS'
                           featureidkey = "properties.CODE_IRIS",
                           color="count",
                           color_continuous_scale="Viridis",
                           range_color=(choropleth_data['count'].min(), choropleth_data['count'].max()),
                           mapbox_style="carto-positron",
                           zoom=5,
                           center={"lat": 46.2276, "lon": 2.2137},
                           labels={'count':'Patient count'}
                          )

# Update the layout of the map
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

# Show the figure
fig